# Feature Maps, Kernels and the QML Bridge
Predict first, run the calculations and explain the differences. Data are generated locally with fixed seeds; reported scores describe these teaching examples, not general performance guarantees.


In [ ]:
%pip -q install "numpy>=2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score


## Check an explicit polynomial feature map
A PSD matrix need not have positive entries, although this squared-dot-product example does.

**Predict:** Compute the 3×3 Gram matrix for (1,0),(0,1),(1,1) from Lesson 12.1. Why is its last diagonal entry four rather than one?


In [ ]:
def phi(X):
    X=np.asarray(X)
    return np.column_stack([X[:,0]**2,np.sqrt(2)*X[:,0]*X[:,1],X[:,1]**2])
X=np.array([[1,2],[2,1],[-1,1]],float)
K=(X@X.T)**2
assert np.allclose(K,phi(X)@phi(X).T)
assert np.linalg.eigvalsh(K).min()>-1e-10
assert np.isclose(K[0,1],16)


## An analytic product-state fidelity kernel
Each scaled coordinate prepares Ry(xj)|0>. Product-state fidelities multiply. This analytic classical calculation is a bridge example, not a quantum hardware experiment or advantage claim.


In [ ]:
def state_kernel(X,Y):
    return np.prod(np.cos((X[:,None,:]-Y[None,:,:])/2)**2,axis=2)
assert np.isclose(state_kernel(np.array([[0.]]),np.array([[np.pi]]))[0,0],0)
Kq=state_kernel(X,X)
assert np.allclose(np.diag(Kq),1)
assert np.linalg.eigvalsh(Kq).min()>-1e-10


## Compare procedures on the same data
All methods see the same feature coordinates and split. Scaling is fitted only within training folds. Search budgets differ and are stated: linear and analytic state kernels use three C values, RBF uses nine C/gamma pairs. This is a teaching comparison, not a definitive benchmark.


In [ ]:
X,y=make_moons(n_samples=240,noise=.2,random_state=2026)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
cv=StratifiedKFold(4,shuffle=True,random_state=42)
models=[("linear",SVC(kernel="linear"),{"svc__C":[.1,1,10]}),("RBF",SVC(kernel="rbf"),{"svc__C":[.1,1,10],"svc__gamma":[.1,1,10]}),("analytic state kernel",SVC(kernel=state_kernel),{"svc__C":[.1,1,10]})]
baseline=DummyClassifier(strategy="most_frequent").fit(Xtr,ytr)
print("Dummy test accuracy:",baseline.score(Xte,yte))
for name,model,grid in models:
    search=GridSearchCV(make_pipeline(StandardScaler(),model),grid,cv=cv,scoring="balanced_accuracy")
    search.fit(Xtr,ytr)
    pred=search.predict(Xte)
    print(name,"CV choice",search.best_params_,"test accuracy",accuracy_score(yte,pred),"balanced accuracy",balanced_accuracy_score(yte,pred))
    assert np.allclose(search.best_estimator_[0].mean_,Xtr.mean(axis=0))


## Exercises
1. Change the noise level while preserving the comparison protocol.
2. Inspect periodic similarities of the analytic state kernel.
3. Explain which costs an actual circuit-based overlap estimate would introduce.
4. Do not select another kernel based on the displayed test scores; use fresh validation or an independently held evaluation dataset for a new comparison.
References: https://scikit-learn.org/stable/modules/svm.html and https://scikit-learn.org/stable/common_pitfalls.html


## SVM dual coefficients and the margin geometry
For the symmetric four-point dataset, alpha=(1/4,1/4,1/4,1/4) is one optimum. The dual optimum is not unique; an independent solver may return different coefficients with the same fitted normal. Reconstruct the normal rather than requiring identical coefficients.


In [ ]:
points=np.array([[-1,-1],[-1,1],[1,-1],[1,1]],float)
labels=np.array([-1,-1,1,1])
fit=SVC(kernel="linear",C=1,tol=1e-10).fit(points,labels)
alpha=np.zeros(4)
alpha[fit.support_]=abs(fit.dual_coef_[0])
normal=(alpha*labels)@points
assert np.allclose(normal,[1,0],atol=1e-5)
assert np.isclose(alpha@labels,0)
assert np.allclose(labels*(points@normal+fit.intercept_[0]),1,atol=1e-5)
print("One fitted optimum:",alpha,"normal:",normal)
# This symmetric dataset has nonunique dual optima; the browser uses
# the feasible symmetric optimum alpha=(1/4,1/4,1/4,1/4).
symmetric=np.full(4,.25)
assert np.allclose((symmetric*labels)@points,normal,atol=1e-5)


## Reproduce the six-point classroom experiment
The browser activity is separable hard-margin. SVC below uses finite C=1000; we explicitly check zero slack and coefficients strictly below C, so its solution also solves the hard-margin problem on these points. Compare the original data with P1 moved to x=0.3. Predict the change in separating width before running.


In [ ]:
classroom = np.array([[1,1],[2,-.5],[2,2],[-1,-1],[-2,.5],[-2,-2]],float)
class_y = np.array([1,1,1,-1,-1,-1])
def fit_classroom(data):
    fitted = SVC(kernel="linear", C=1000, tol=1e-10).fit(data,class_y)
    a = np.zeros(6); a[fitted.support_] = abs(fitted.dual_coef_[0])
    w = (a*class_y) @ data; bias = fitted.intercept_[0]
    margins = class_y*(data@w+bias)
    assert np.isclose(a@class_y,0,atol=1e-7)
    assert margins.min() >= 1-1e-6 and a.max() < 1000-1
    return w,bias,a,2/np.linalg.norm(w)
original = fit_classroom(classroom)
moved = classroom.copy(); moved[0,0] = .3
changed = fit_classroom(moved)
assert np.allclose(original[0],[.6,.4],atol=1e-5)
assert changed[3] < original[3]
for label,result in [("Original",original),("P1 moved",changed)]:
    print(label,"normal",result[0],"bias",result[1],"alpha",result[2],"width",result[3])
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2,figsize=(10,4))
xx,yy=np.meshgrid(np.linspace(-2.5,2.5,100),np.linspace(-2.5,2.5,100))
for ax,data,result,title in zip(axes,[classroom,moved],[original,changed],["Original data","P1 moved inward"]):
    w,bias,a,width=result
    ax.contour(xx,yy,w[0]*xx+w[1]*yy+bias,levels=[-1,0,1],colors=["grey","black","grey"],linestyles=["--","-","--"])
    for label,marker in [(1,"o"),(-1,"s")]:
        mask=class_y==label;ax.scatter(data[mask,0],data[mask,1],marker=marker,label=f"Class {label}")
    active=a>1e-6
    ax.scatter(data[active,0],data[active,1],s=180,facecolors="none",edgecolors="black")
    ax.set(title=f"{title}: width={width:.3f}",xlabel="x1",ylabel="x2",aspect="equal");ax.legend()
fig.tight_layout();plt.show()


## Explain the result
Identify points with nonzero alpha before and after movement. Width falls because the inward point constrains the separating strip more tightly. Distinguish this fitted result from manually adjusted feasible weights. For a quantum kernel, the same learner would consume different pairwise similarities; that alone establishes neither predictive nor computational advantage.
